# Question 2 / Phase 2: Package Prioritization and Box Filling

This notebook contains the second question/phase of the Mini-Courier Planner capstone assignment.

## Theory

Phase 2 uses sorting and dynamic programming.

Parcels are first sorted by priority and deadline. Then 0/1 knapsack dynamic programming is used to choose the best parcel combination within vehicle capacity.

The recurrence is:

```text
dp[i][w] = max(dp[i - 1][w], benefit[i - 1] + dp[i - 1][w - weight[i - 1]])
```

Time complexity is `O(n * capacity)`.

Space complexity is `O(n * capacity)`.

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple


@dataclass(frozen=True)
class Parcel:
    parcel_id: str
    destination: str
    weight: int
    value: int
    deadline: int
    priority: str


PRIORITY_SCORE = {
    "High": 3,
    "Medium": 2,
    "Low": 1,
}


def priority_score(parcel: Parcel) -> int:
    return PRIORITY_SCORE[parcel.priority]


def parcel_benefit(parcel: Parcel) -> int:
    urgency_bonus = max(0, 6 - parcel.deadline) * 5
    priority_bonus = priority_score(parcel) * 20
    return parcel.value + priority_bonus + urgency_bonus


parcels = [
    Parcel("P01", "A", 4, 60, 2, "High"),
    Parcel("P02", "B", 2, 35, 1, "High"),
    Parcel("P03", "C", 6, 80, 4, "Medium"),
    Parcel("P04", "D", 3, 45, 3, "High"),
    Parcel("P05", "E", 5, 70, 2, "Medium"),
    Parcel("P06", "F", 1, 20, 1, "Low"),
    Parcel("P07", "G", 7, 100, 5, "High"),
    Parcel("P08", "H", 2, 25, 3, "Low"),
    Parcel("P09", "I", 2, 30, 3, "High"),
]

VEHICLE_CAPACITY = 12

In [ ]:
def sort_parcels_by_priority_and_deadline(parcel_list: List[Parcel]) -> List[Parcel]:
    return sorted(
        parcel_list,
        key=lambda parcel: (-priority_score(parcel), parcel.deadline),
    )


sorted_parcels = sort_parcels_by_priority_and_deadline(parcels)

print("Sorted parcels by priority and deadline:")
for parcel in sorted_parcels:
    print(
        f"{parcel.parcel_id} | destination={parcel.destination} | "
        f"priority={parcel.priority:<6} | deadline={parcel.deadline} | "
        f"weight={parcel.weight} | benefit={parcel_benefit(parcel)}"
    )

In [ ]:
def knapsack_select_parcels(
    parcel_list: List[Parcel], capacity: int
) -> Tuple[int, List[Parcel], List[List[int]]]:
    n = len(parcel_list)
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    for i in range(1, n + 1):
        parcel = parcel_list[i - 1]
        weight = parcel.weight
        benefit = parcel_benefit(parcel)

        for current_capacity in range(capacity + 1):
            without_parcel = dp[i - 1][current_capacity]
            with_parcel = 0

            if weight <= current_capacity:
                with_parcel = benefit + dp[i - 1][current_capacity - weight]

            dp[i][current_capacity] = max(without_parcel, with_parcel)

    selected = []
    remaining_capacity = capacity

    for i in range(n, 0, -1):
        if dp[i][remaining_capacity] != dp[i - 1][remaining_capacity]:
            parcel = parcel_list[i - 1]
            selected.append(parcel)
            remaining_capacity -= parcel.weight

    selected.reverse()
    return dp[n][capacity], selected, dp


maximum_benefit, selected_parcels, dp_table = knapsack_select_parcels(
    sorted_parcels,
    VEHICLE_CAPACITY,
)

total_weight = sum(parcel.weight for parcel in selected_parcels)
total_benefit = sum(parcel_benefit(parcel) for parcel in selected_parcels)
remaining_capacity = VEHICLE_CAPACITY - total_weight

print(f"Vehicle capacity: {VEHICLE_CAPACITY} kg")
print(f"Maximum benefit from DP table: {maximum_benefit}")
print("Selected parcels:", ", ".join(parcel.parcel_id for parcel in selected_parcels))
print(f"Total selected weight: {total_weight} kg")
print(f"Total selected benefit: {total_benefit}")
print(f"Remaining capacity: {remaining_capacity} kg")

In [ ]:
print(f"DP table rows: {len(dp_table)}")
print(f"DP table columns: {len(dp_table[0])}")
print("Last DP row:", dp_table[-1])

## Test Cases and Verification

In [ ]:
# Test 1: Normal case
assert total_weight <= VEHICLE_CAPACITY
assert maximum_benefit == total_benefit == 550
print("Normal case passed: selected parcels fit capacity and reach optimum benefit.")

# Test 2: Capacity edge case
small_capacity_benefit, small_capacity_selection, _ = knapsack_select_parcels(
    sorted_parcels,
    capacity=3,
)
assert small_capacity_benefit == 185
assert sum(parcel.weight for parcel in small_capacity_selection) <= 3
print("Small capacity edge case passed.")

# Test 3: Stable sorting tie case
sorted_ids = [parcel.parcel_id for parcel in sorted_parcels]
assert sorted_ids.index("P04") < sorted_ids.index("P09")
print("Stable sorting tie case passed: P04 remains before P09.")

## Phase 2 Summary

Phase 2 sorts parcels by priority and deadline, then uses 0/1 knapsack dynamic programming to select the best parcels within the vehicle capacity.